# Model Training, Tuning & Explainability

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

X_train = pd.read_parquet('../data/X_train.parquet')
X_test = pd.read_parquet('../data/X_test.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')['Churn']
y_test = pd.read_parquet('../data/y_test.parquet')['Churn']

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

In [ ]:
SEED = 42

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
    'Random Forest': RandomForestClassifier(random_state=SEED),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
    })
    print(f'--- {name} ---')
    print(classification_report(y_test, y_pred))

pd.DataFrame(results).round(4)

### XGBoost (requires install)

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(n_estimators=100, random_state=SEED, eval_metric='logloss', use_label_encoder=False)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

results.append({
    'Model': 'XGBoost',
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_prob),
})
print(classification_report(y_test, y_pred))
pd.DataFrame(results).round(4)

### Hyperparameter Tuning (Random Forest)

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
}

rf = RandomForestClassifier(random_state=SEED)
grid = GridSearchCV(rf, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print('Best params:', grid.best_params_)
print('Best CV ROC-AUC:', round(grid.best_score_, 4))

In [ ]:
best_rf = grid.best_estimator_
y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:, 1]

results.append({
    'Model': 'RF (tuned)',
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_prob),
})
print(classification_report(y_test, y_pred))
pd.DataFrame(results).round(4)

### Model Explainability (SHAP)

In [ ]:
import shap

explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)

# Summary plot
shap.summary_plot(shap_values[:,:,1], X_test, plot_type='bar', show=False)
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()

In [ ]:
# Waterfall for a single prediction (customer index 0)
shap.plots.waterfall(shap.Explanation(values=shap_values[0,:,1], 
                                      base_values=explainer.expected_value[1],
                                      data=X_test.iloc[0], feature_names=X_test.columns.tolist()))
plt.show()

In [ ]:
# Save best model
import joblib
joblib.dump(best_rf, '../models/churn_model.pkl')
print('Model saved to models/churn_model.pkl')